# 计算图和泳道图

这一节把视角从“怎么写 kernel”推进到“怎么分析和复用 kernel 的执行”。



## 1. 环境准备


In [ ]:
import os
os.environ['TILE_FWK_DEVICE_ID'] = '0'
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'
import torch
import pypto
import numpy as np
from numpy.testing import assert_allclose
import torch_npu


def get_device():
    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    return f"npu:{device_id}"


device = get_device()
RUN_MODE = pypto.RunMode.NPU


print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<not set>"))
print("device:", device)
print("run_mode:", RUN_MODE)
print("pypto:", pypto.__file__)


## 2. 代码展示

使用PyPTO框架实现一个简单的算子，并通过测试用例验证其正确性。本节可以先初步了解计算图和泳道图，随着课程学完，在后续的开发工作中真正实践。

可以暂时不了解softmax_core的数学含义，通过本节的学习，了解如何使用PyPTO的API来构建自定义算子。并且在程序完成后，通过`PyPTO Toolkit`可视化工具查看计算图结构，并观测算子的各项性能数据。


In [ ]:
def softmax_core(x: pypto.Tensor) -> pypto.Tensor:
    row_max = pypto.amax(x, dim=-1, keepdim=True)
    sub = x - row_max
    exp = pypto.exp(sub)
    esum = pypto.sum(exp, dim=-1, keepdim=True)
    return exp / esum


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def softmax_kernel(
    input_tensor: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output_tensor: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
):
    bs, seqlen, head, dim = input_tensor.shape
    tile_b = 1
    b_loop = bs // tile_b

    # Tiling shape setting for efficient execution
    pypto.set_vec_tile_shapes(1, 4, 1, 64)

    for idx in pypto.loop(0, b_loop, 1, name="LOOP_L0_bIdx", idx_name="idx"):
        b_offset = idx * tile_b
        b_offset_end = (idx + 1) * tile_b
        input_view = input_tensor[b_offset:b_offset_end, :seqlen, :head, :dim]
        softmax_out = softmax_core(input_view)
        output_tensor[b_offset:b_offset_end, ...] = softmax_out


def test_softmax(device_id: int = None, dynamic: bool = True) -> None:
    device = get_device()
    shape = (32, 32, 1, 256)
    x = torch.rand(shape, dtype=torch.float, device=device)
    y = torch.zeros(shape, dtype=torch.float, device=device)
    softmax_kernel(x, y)
    golden = torch.softmax(x, dim=-1).cpu()
    y = y.cpu()
    max_diff = np.abs(y.numpy() - golden.numpy()).max()
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {y.shape}")
    print(f"Max difference: {max_diff:.6f}")
    assert_allclose(np.array(y), np.array(golden), rtol=3e-3, atol=3e-3)
    print("✓ Softmax test passed")
    print()


test_softmax()


## 3. 编译与执行


### 3.1 查看计算图

开启图编译阶段调试模式开关。  

@pypto.frontend.jit(
    debug_options={"runtime_debug_mode": 1}
)

PyPTO程序在编译过程中，会自动生成由Tensor和Operation组合而成的图结构，即计算图。该计算图经过PyPTO编译优化流程，完成从原始计算图到可执行图的编译过程，最终生成可在昇腾硬件环境中运行的可执行代码，以实现实际的计算任务。用户可借助PyPTO Toolkit可视化工具可以直观地查看计算图结构，了解计算图的节点信息，更便捷地进行算子功能调试。

右键单击${work_path}/output/output_*/program.json文件，在弹出的菜单中选择“使用PyPTO Toolkit打开”。

program.json文件包含了Execute Graph和Block Graph的汇总信息，图中的关键信息为：左右两边的卡片为Tensor节点（代表输入/输出数据）、中间卡片为调用节点（带有fx标识，单击可以实现信息钻取）。



<img src="./images/tensor_graph.png" alt="tensor_graph"  width="900px" >

双击中间卡片逐层钻取到下图所示的Execute Graph。

<img src="./images/execute_graph.png" alt="execute_graph"  width="900px" >

双击上图的调用节点，可以看到Block Graph子图信息，标识着任务的具体执行过程。放大后可以看到图中具体的Tensor和Operation节点信息和连接关系，例如图中可以看到SUB和EXP操作。

<img src="./images/block_graph.png" alt="block_graph"  width="900px" >


### 3.2 查看泳道图

泳道图用于直观展示计算图的实际调度与执行过程，清晰呈现任务的执行顺序和耗时信息，帮助开发者分析算子性能瓶颈。下面将介绍如何采集泳道图数据，并通过PyPTO Toolkit查看泳道图。

通过给@pypto.frontend.jit装饰器的入参debug_options配置图执行阶段调试开关启动性能数据采集功能。重新执行代码可以生成泳道图


@pypto.frontend.jit(debug_options={"runtime_debug_mode": 1})

在${work_path}/output/output_*/目录（*代表时间戳）下生成泳道图数据文件，文件名为：merged_swimlane.json。右键单击merged_swimlane.json，在弹出的菜单中选择“使用PyPTO Toolkit打开”，如下图所示。

<img src="./images/swimlane.png" alt="swimlane"  width="900px" >

上图中带有色块的部分即为泳道，展示了每个AIC/AIV上的任务执行情况。泳道条目的长度对应任务的耗时，能够直观地反映计算的密集程度。用户可以通过观察相邻泳道之间的空闲间隔以及耗时较长的泳道条目，来分析可能存在的性能瓶颈问题。

## 4. 课后练习

本节练习用于复盘计算图和泳道图的作用。请完成以下题目。

1. （选择题）关于 PyPTO 计算图的查看方式，以下说法正确的是？  
A. 计算图数据存储在 merged_swimlane.json 文件中，需要用文本编辑器打开  
B. program.json 文件包含了 Execute Graph 和 Block Graph 的汇总信息，通过右键选择"使用 PyPTO Toolkit 打开"查看  
C. 计算图只能通过命令行工具 pypto-cli 导出为 SVG 图片查看  
D. 计算图在程序运行时直接在终端以文本形式输出，无需额外文件  

2. （选择题）要采集泳道图性能数据，需要在 @pypto.frontend.jit 装饰器中做怎样的配置？  
A. 在 runtime_options 中设置 {"run_mode": pypto.RunMode.SIM}  
B. 在 pass_options 中设置 {"enable_swimlane": True}  
C. 在 debug_options 中设置 {"runtime_debug_mode": 1}  
D. 在 codegen_options 中设置 {"output_swimlane": True}  

3. （选择题）关于泳道图的作用与特征，以下描述正确的是？  
A. 泳道图用于展示计算图的编译优化过程，每个泳道代表一个编译 Pass  
B. 泳道图中的色块代表 AIC/AIV 上的任务执行情况，泳道条目长度对应任务耗时，可用于分析性能瓶颈  
C. 泳道图和计算图查看的是同一个 program.json 文件，只是展示视角不同  
D. 泳道图只能在 NPU 硬件上生成，SIM 模式下无法生成泳道数据  
 
**执行以下代码获取答案。**


In [ ]:
!cat ./answer/02.04_answer.txt
